# Decision Tree Development Notebook (v1) 

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from functools import reduce
from typing import Literal

In [ ]:
def prod(arr: list) -> float:
    """Computes the product of all elevents of a given vector"""
    return reduce(lambda x, y: x * y, arr, 1)

def gini(arr: list) -> float:
    """Computes Gini Impurity for a given label vector"""
    n = len(arr)
    value_counts = dict(Counter(arr))
    unique_categories = list(value_counts.keys())
    prob_list = {cat: (value_counts[cat] / n) for cat in unique_categories}
    return 1 - sum(p**2 for p in prob_list.values())

def get_cutoff(feature, y, base_gini, return_gain: bool = False):
    """Searches a good cutoff in a feature vector in order to maximize node purity gain"""
    feature_nrows_unique = len(set(feature))
    cuts = int(np.floor(np.min([feature_nrows_unique, 10])))
    candidates = list(set([np.quantile(feature, q=q).item() for q in np.linspace(0, 1, cuts)]))
    curr_winner, curr_winner_gain = None, 0
    for candidate in candidates:
        left_gini, right_gini = gini(y[feature < candidate]), gini(y[feature >= candidate])
        gain = (base_gini - left_gini) + (base_gini - right_gini)  # TODO: Generalize
        if gain > curr_winner_gain:
            curr_winner_gain, curr_winner = gain, candidate
    return curr_winner if return_gain is False else (curr_winner, curr_winner_gain)

In [87]:
# Root node -> Populate in function of a certain feature (index) -> Search 

class BinaryTree:
    """Manual implementation of a BinaryTree"""

    def __init__(self):
        # Starts from root node (which is also a tree by itself)
        self.left = None  # Leaf (or "< Node")
        self.right = None  # Leaf (or ">= Node")
        # State
        self.feature_index: int = None  # Index of the feature used to split current node
        self.cutoff_value: float = None  # Boundary
        self.gini = None

    def _set_child(self, child, direction: Literal["left", "right"]) -> None:
        assert direction in ["left", "right"], "<direction> value must be either left or right"
        if direction == "left":
            self.left = child
        else:
            self.right = child

    def __repr__(self):
        line1 = "<BinaryTree object>"
        line2 = f"Left child = {self.left}"
        line3 = f"Right child = {self.right}"
        line4 = f"Feature index (on train data) = {self.feature_index}"
        line5 = f"Cutoff value in feature vector = {self.cutoff_value}"
        line6 = f"Impurity = {self.gini}"
        final_str = f"{line1}\n- {line2}\n- {line3}\n- {line4}\n- {line5}\n- {line6}"
        return final_str

In [88]:
tree = BinaryTree()
tree

<BinaryTree object>
- Left child = None
- Right child = None
- Feature index (on train data) = None
- Cutoff value in feature vector = None
- Impurity = None

In [ ]:
class DecisionTreeClassifier:
    """Manual implementation of DecisionTreeClassifier (v1, i.e. binary & no relevant hyperparameters)"""

    def __init__(self):
        self.tree = BinaryTree()
        self.trained = False

    def train(self, X, y) -> None:
        """
        Executes the training process, note that this is an inplace logic
        Reminder: stop criteria is gini_of_node = 0
        """
        if self.trained is True:
            raise ValueError("Tree is already trained")

        # TODO: recursive sub-training process till impurities are zero
        # Root gini (overall)
        overall_impurity = gini(y)
        n_features = X.shape[1]

        purity_gains = []
        for feature_index in range(n_features):
            cutoff_value, purity_gain = get_cutoff(X[:, feature_index], y, base_gini=overall_impurity, return_gain=True)
            purity_gains.append((feature_index, cutoff_value, purity_gain))
        best_feature_index, best_cutoff_value, _ = max(purity_gains, key=lambda t: t[2])

        self.tree.feature_index = best_feature_index
        self.tree.cutoff_value = best_cutoff_value
        self.tree.gini = overall_impurity

        # child = BinaryTree()
        # self._set_child()
        
        # Iterate over features and select the one that maximizes absolute gini difference
        

        self.trained = True

    # def _single_inference(self, X) -> float:
    #     raise NotImplementedError

    # def inference(self, X) -> float:
    #     raise NotImplementedError


In [108]:
dtc = DecisionTreeClassifier()

In [109]:
data = pd.read_parquet("../example_data/iris_dataset.parquet")
data

,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [110]:
dtc.train(data.drop(columns=["target"]).values, data["target"].values)

In [111]:
dtc.tree

<BinaryTree object>
- Left child = None
- Right child = None
- Feature index (on train data) = 2
- Cutoff value in feature vector = 2.6333333333333306
- Impurity = 0.6666666666666667